# Phase 1 — Supervised Learning — Regression

**Companion Colab notebook to the study guide PDF.**

This notebook implements every method from the poster, end to end:

1. **Linear Regression** — normal equation, gradient descent (from scratch) + scikit-learn
2. **Polynomial Regression** — feature expansion + an overfitting demo
3. **Ridge Regression (L2)** — smooth shrinkage + regularization path
4. **Lasso Regression (L1)** — automatic feature selection + path to zero
5. **Elastic Net** — combining L1 and L2

Each section reproduces the *exact hand-solved numbers* from the PDF, then shows the same idea on
a larger, more realistic dataset. Run the cells top to bottom (Runtime ▸ Run all).


## 0. Setup

In [ ]:
# Standard scientific Python stack — all preinstalled on Colab
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet,
    RidgeCV, LassoCV, ElasticNetCV,
)
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

np.set_printoptions(precision=4, suppress=True)
np.random.seed(42)
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
print("Setup complete.")

## 1. Linear Regression  `REGRESSION`

Fits a straight line $\hat{y} = \theta_0 + \theta_1 x$ by minimising the mean squared error
$J(\theta) = \frac{1}{2m}\sum (\hat y^{(i)} - y^{(i)})^2$.

We first reproduce the **hand-solved example** from the PDF:
$x=(1,2,3,4,5)$, $y=(2,4,5,4,5)$, which should give $\hat y = 2.2 + 0.6x$ and MSE $=0.48$.

In [ ]:
# --- The exact dataset from the PDF worked example ---
x = np.array([1, 2, 3, 4, 5], dtype=float)
y = np.array([2, 4, 5, 4, 5], dtype=float)

# Method A: the closed-form least-squares formulas
x_bar, y_bar = x.mean(), y.mean()
Sxx = np.sum((x - x_bar) ** 2)
Sxy = np.sum((x - x_bar) * (y - y_bar))
theta1 = Sxy / Sxx
theta0 = y_bar - theta1 * x_bar
print(f"Closed form:  theta0 = {theta0:.4f}, theta1 = {theta1:.4f}")

y_hat = theta0 + theta1 * x
mse = np.mean((y_hat - y) ** 2)
print(f"Predictions:  {y_hat}")
print(f"MSE        :  {mse:.4f}   (PDF says 0.48)")

In [ ]:
# Method B: the Normal Equation in matrix form  theta = (X^T X)^-1 X^T y
X = np.column_stack([np.ones_like(x), x])     # add a bias column of 1s
theta = np.linalg.inv(X.T @ X) @ X.T @ y
print(f"Normal equation: theta0 = {theta[0]:.4f}, theta1 = {theta[1]:.4f}")

In [ ]:
# Method C: Gradient Descent from scratch
def gradient_descent(x, y, alpha=0.01, n_iters=10000):
    m = len(x)
    t0, t1 = 0.0, 0.0
    history = []
    for _ in range(n_iters):
        y_hat = t0 + t1 * x
        err = y_hat - y
        grad0 = err.mean()
        grad1 = (err * x).mean()
        t0 -= alpha * grad0
        t1 -= alpha * grad1
        history.append(0.5 * np.mean(err ** 2))
    return t0, t1, history

# One step (matches the PDF: theta0 -> 0.04, theta1 -> 0.132)
t0_1, t1_1, _ = gradient_descent(x, y, alpha=0.01, n_iters=1)
print(f"After 1 step :  theta0 = {t0_1:.4f}, theta1 = {t1_1:.4f}  (PDF: 0.04, 0.132)")

# Full convergence
t0, t1, hist = gradient_descent(x, y, alpha=0.01, n_iters=10000)
print(f"After 10k    :  theta0 = {t0:.4f}, theta1 = {t1:.4f}  (-> 2.2, 0.6)")

In [ ]:
# Visualise the fit and the cost going down
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))

ax[0].scatter(x, y, color="navy", zorder=3, label="data")
xs = np.linspace(0, 6, 100)
ax[0].plot(xs, theta0 + theta1 * xs, color="crimson", label=rf"$\hat y={theta0:.1f}+{theta1:.1f}x$")
ax[0].set_title("Linear fit"); ax[0].set_xlabel("x"); ax[0].set_ylabel("y"); ax[0].legend()

ax[1].plot(hist, color="midnightblue")
ax[1].set_title("Gradient descent: cost vs iteration")
ax[1].set_xlabel("iteration"); ax[1].set_ylabel("J(theta)")
plt.tight_layout(); plt.show()

In [ ]:
# Method D: scikit-learn (what you'll actually use day to day)
lin = LinearRegression().fit(x.reshape(-1, 1), y)
print(f"sklearn      :  intercept = {lin.intercept_:.4f}, slope = {lin.coef_[0]:.4f}")
print(f"R^2          :  {lin.score(x.reshape(-1, 1), y):.4f}")

## 2. Polynomial Regression  `REGRESSION`

Add powers of $x$ as new features: $\hat y = \theta_0 + \theta_1 x + \theta_2 x^2 + \dots$
It's still *linear regression*, just on transformed features.

First, reproduce the PDF example: a quadratic through $(0,1),(1,0),(2,3)$ recovers
$\hat y = 2x^2 - 3x + 1$.

In [ ]:
# Exact quadratic through 3 points (PDF worked example)
xp = np.array([0, 1, 2], dtype=float)
yp = np.array([1, 0, 3], dtype=float)

Xpoly = np.column_stack([np.ones_like(xp), xp, xp**2])   # [1, x, x^2]
coef = np.linalg.solve(Xpoly, yp)
print(f"theta0, theta1, theta2 = {coef}   (PDF: 1, -3, 2)")
print("=> y = 2x^2 - 3x + 1")

### Overfitting demo: why high degree is dangerous

We generate noisy data from a smooth curve, then fit polynomials of increasing degree. Low degree
**underfits** (high bias); very high degree **overfits** (high variance) — it chases the noise and
generalises poorly.

In [ ]:
# Noisy data from a true function f(x) = 0.5x^3 - x^2 + 2
rng = np.random.RandomState(0)
n = 30
x_raw = np.sort(rng.uniform(-3, 3, n))
true_f = lambda t: 0.5*t**3 - t**2 + 2
y_raw = true_f(x_raw) + rng.normal(0, 3, n)

x_tr, x_te, y_tr, y_te = train_test_split(x_raw, y_raw, test_size=0.4, random_state=1)

degrees = [1, 3, 9, 15]
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
grid = np.linspace(-3.2, 3.2, 300)

for ax, d in zip(axes, degrees):
    model = make_pipeline(PolynomialFeatures(d), LinearRegression())
    model.fit(x_tr.reshape(-1, 1), y_tr)
    train_mse = mean_squared_error(y_tr, model.predict(x_tr.reshape(-1, 1)))
    test_mse  = mean_squared_error(y_te, model.predict(x_te.reshape(-1, 1)))
    ax.scatter(x_tr, y_tr, s=25, color="navy", label="train")
    ax.scatter(x_te, y_te, s=25, color="orange", marker="^", label="test")
    ax.plot(grid, model.predict(grid.reshape(-1, 1)), color="crimson")
    ax.set_ylim(y_raw.min()-5, y_raw.max()+5)
    ax.set_title(f"degree {d}\ntrain MSE={train_mse:.1f}  test MSE={test_mse:.1f}")
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print("Notice: training error keeps falling with degree, but TEST error is lowest at a moderate degree.")

## 3. Ridge Regression (L2)  `REGRESSION`

Adds an L2 penalty $\lambda\sum\theta_j^2$. Coefficients **shrink smoothly toward zero but never
reach exactly zero**.

Reproduce the PDF shrinkage table on $x=(1,2,3)$, $y=(1,2,2)$ (single feature, no intercept):
$\theta = \dfrac{\rho}{z+\lambda} = \dfrac{11}{14+\lambda}$.

In [ ]:
xr = np.array([1, 2, 3], dtype=float)
yr = np.array([1, 2, 2], dtype=float)
z = np.sum(xr**2)        # 14
rho = np.sum(xr*yr)      # 11
print(f"z = {z:.0f}, rho = {rho:.0f}")
print("lambda   theta")
for lam in [0, 1, 10, 14, 100]:
    print(f"{lam:>5}   {rho/(z+lam):.4f}")
print("=> shrinks toward 0 but never hits exactly 0")

### Ridge regularization path on a multi-feature dataset

As $\lambda$ grows, every coefficient is pulled toward zero together.

In [ ]:
# Build a dataset with several correlated features
from sklearn.datasets import make_regression
Xm, ym = make_regression(n_samples=80, n_features=8, n_informative=4,
                         noise=15.0, random_state=3)
Xm = StandardScaler().fit_transform(Xm)   # standardise before regularising!

alphas = np.logspace(-2, 3, 60)
ridge_coefs = []
for a in alphas:
    ridge_coefs.append(Ridge(alpha=a).fit(Xm, ym).coef_)
ridge_coefs = np.array(ridge_coefs)

plt.figure()
for j in range(Xm.shape[1]):
    plt.plot(alphas, ridge_coefs[:, j], label=f"feat {j}")
plt.xscale("log")
plt.xlabel("lambda (alpha)"); plt.ylabel("coefficient value")
plt.title("Ridge path: all coefficients shrink, none hit exactly zero")
plt.legend(ncol=2, fontsize=8); plt.show()

## 4. Lasso Regression (L1)  `REGRESSION`

Adds an L1 penalty $\lambda\sum|\theta_j|$. The key difference: coefficients can be driven to
**exactly zero** — automatic feature selection.

Reproduce the PDF: same data as Ridge, soft-threshold
$\theta = \dfrac{\max(\rho-\lambda,\,0)}{z}$, which hits zero at $\lambda = \rho = 11$.

In [ ]:
print("lambda   theta")
for lam in [0, 5, 10, 11, 14]:
    theta = max(rho - lam, 0) / z
    print(f"{lam:>5}   {theta:.4f}")
print("=> EXACTLY zero from lambda = 11 onward (Ridge never did this)")

### Lasso path: watch coefficients snap to zero

Compare directly with the Ridge path above — here individual lines reach (and stay at) exactly zero,
selecting a subset of features.

In [ ]:
lasso_alphas = np.logspace(-2, 1.5, 60)
lasso_coefs = []
for a in lasso_alphas:
    lasso_coefs.append(Lasso(alpha=a, max_iter=10000).fit(Xm, ym).coef_)
lasso_coefs = np.array(lasso_coefs)

plt.figure()
for j in range(Xm.shape[1]):
    plt.plot(lasso_alphas, lasso_coefs[:, j], label=f"feat {j}")
plt.xscale("log")
plt.xlabel("lambda (alpha)"); plt.ylabel("coefficient value")
plt.title("Lasso path: coefficients hit exactly zero (feature selection)")
plt.legend(ncol=2, fontsize=8); plt.show()

# How many features survive at a moderate lambda?
lasso = Lasso(alpha=1.0, max_iter=10000).fit(Xm, ym)
print("Coefficients at lambda=1.0:", lasso.coef_)
print("Non-zero (selected) features:", np.sum(lasso.coef_ != 0), "out of", Xm.shape[1])

## 5. Elastic Net  `REGRESSION`

Combines both penalties: $\lambda_1\sum|\theta_j| + \lambda_2\sum\theta_j^2$.
Single-feature update: $\theta = \dfrac{S(\rho,\lambda_1)}{z+\lambda_2}$.

Reproduce the PDF comparison on the same data with $\lambda_1=5$, $\lambda_2=14$.

In [ ]:
lam1, lam2 = 5, 14
soft = max(rho - lam1, 0)                 # S(rho, lambda1) for rho>0
enet_theta  = soft / (z + lam2)
lasso_theta = max(rho - lam1, 0) / z      # L1 only
ridge_theta = rho / (z + lam2)            # L2 only

print(f"Lasso  (l1=5)      : {lasso_theta:.4f}")
print(f"Ridge  (l2=14)     : {ridge_theta:.4f}")
print(f"ElasticNet (5, 14) : {enet_theta:.4f}   (smallest; combines both)")

### scikit-learn note on the Elastic Net parameterisation

scikit-learn uses `alpha` (overall strength) and `l1_ratio` (the mix between L1 and L2):
`l1_ratio=1.0` is pure Lasso, `l1_ratio=0.0` is pure Ridge.

In [ ]:
enet = ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000).fit(Xm, ym)
print("Elastic Net coefficients:", enet.coef_)
print("Non-zero:", np.sum(enet.coef_ != 0), "out of", Xm.shape[1])

## 6. Putting it all together

Compare every method on the same train/test split, and let cross-validation pick the regularisation
strength $\lambda$ automatically (the right way to choose it in practice).

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(Xm, ym, test_size=0.3, random_state=7)

models = {
    "Linear":      LinearRegression(),
    "Ridge (CV)":  RidgeCV(alphas=np.logspace(-2, 3, 50)),
    "Lasso (CV)":  LassoCV(alphas=np.logspace(-2, 2, 50), max_iter=10000),
    "ElasticNet (CV)": ElasticNetCV(l1_ratio=[.2, .5, .8, .95],
                                    alphas=np.logspace(-2, 2, 50), max_iter=10000),
}

print(f"{'Model':<18}{'Test MSE':>10}{'Test R^2':>10}{'Non-zero coefs':>16}")
print("-" * 54)
for name, mdl in models.items():
    mdl.fit(Xtr, ytr)
    pred = mdl.predict(Xte)
    mse = mean_squared_error(yte, pred)
    r2 = r2_score(yte, pred)
    nz = np.sum(np.abs(mdl.coef_) > 1e-8)
    print(f"{name:<18}{mse:>10.2f}{r2:>10.3f}{nz:>16}")

### Takeaways

- **Linear** is the baseline — understand its cost and gradient descent first.
- **Polynomial** adds curve-fitting power but invites overfitting at high degree.
- **Ridge (L2)** shrinks all weights smoothly; great with many correlated, useful features.
- **Lasso (L1)** zeros out weights → automatic feature selection and sparse, interpretable models.
- **Elastic Net** blends both — a strong default when features are numerous and correlated.

**Always standardise features** before Ridge/Lasso/Elastic Net, and **choose $\lambda$ by
cross-validation** (the `...CV` estimators above do this for you).
